In [ ]:
!pip install leven

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from sklearn.cluster import DBSCAN
from sklearn.feature_extraction.text import TfidfVectorizer
import re
import numpy as np
import sklearn

In [2]:
def extract_require_statements(parquet_file_path):
    df = pd.read_parquet(parquet_file_path)
    if 'failure_invariant' not in df.columns:
        raise ValueError("The Parquet file does not contain a 'failure_invariant' column.")
    require_statements = []
    for invariant in df['failure_invariant']:
        matches = re.findall(r'\brequire\b.*', str(invariant))
        require_statements.extend(matches)
    return require_statements


In [3]:
def dbscan_cluster(invariants, eps=0.5, min_samples=5, metric = 'cosine'):
    """
    DBSCAN with cosine 
    """
    vectorizer = TfidfVectorizer()
    invariant_vectors = vectorizer.fit_transform(invariants)
    db = DBSCAN(eps=eps, min_samples=min_samples, metric=metric)
    db.fit(invariant_vectors)
    cluster_dict = {}
    for label, invariant in zip(db.labels_, invariants):
        if label == -1:
            continue
        if label not in cluster_dict:
            cluster_dict[label] = []
        cluster_dict[label].append(invariant)
    
    return cluster_dict


In [ ]:
parquet_file_path = '../../transactions.parquet'
require_statements = extract_require_statements(parquet_file_path)
print(len(require_statements))
require_statements = list(set(require_statements))
sorted(sklearn.neighbors.VALID_METRICS_SPARSE['brute']) 

labels = dbscan_cluster(require_statements, eps=0.7, min_samples=2, metric= 'euclidean')

for cluster_label, statements in labels.items():
    print(f"Cluster {cluster_label}:")
    counter = 0
    for statement in statements:
        if counter > 3:
            break
        counter += 1
        print(f"  {statement}")
    print("\n")

5764
Cluster 0:
  require(sellCount < 3, "Only 3 sells per block!");
  require(sellCount < 4, "Only 4 sells per block!");
  require(sellCount < 2, "Only 2 sells per block!");
  require(sellCount < 5, "Only 5 sells per block!");


Cluster 1:
  require(_value <= balances[msg.sender], "There is no enough balance.");
  require(_value <= balances[_from], "There is no enough balance.");


Cluster 2:
  require( _isExcludedFromFee[to] || _isExcludedFromFee[from], "trading not yet open" );
  require(_isExcludedFromFee[from] || _isExcludedFromFee[to], "Trading is not active.");
  require( _isExcludedFromFee[from] || _isExcludedFromFee[to], "Trading is not active." );
  require( _isExcludedFromFee[from] || _isExcludedFromFee[to], "Trading is not open yet and you are not authorized" );


Cluster 3:
  require(msg.value >= usdcToEth(Pair.sellTotalFee()), 'EtherVistaRouter: INSUFFICIENT_ETH_FEE');
  require(msg.value > usdcToEth(Pair.buyTotalFee()), 'EtherVistaRouter: INSUFFICIENT_ETH_FEE');


Cluste

In [ ]:
# clustering: only predicates
# find best dinstance function --> semantic distance!!!
# then cluster based on that
# clustering programs (ie: clone detection) 
# use server for clustering

- BERT incoding distance
- jaccard distance, levensthein
- SYNDi 
- other SEMANTIC distances (NL)

In [ ]:
from leven import levenshtein       
import numpy as np
from sklearn.cluster import DBSCAN

require_statements = extract_require_statements(parquet_file_path)
#require_statements = list(set(require_statements))  # Remove duplicates

print(len(require_statements))
def lev_metric(x, y):
    # x and y are arrays with shape (1,) containing indices
    i, j = int(x[0]), int(y[0])     # extract indices
    return levenshtein(require_statements[i], require_statements[j])

# Create X with indices of the require_statements
X = np.arange(len(require_statements)).reshape(-1, 1)

# Apply DBSCAN with the custom Levenshtein distance
db = DBSCAN(eps=10, min_samples=2, metric=lev_metric)
db.fit(X)

# db.labels_ contains the cluster labels for each statement
labels = db.labels_
clusters = {}
for i, label in enumerate(labels):
    if label != -1:  # Ignore noise points (-1)
        if label not in clusters:
            clusters[label] = []
        clusters[label].append(require_statements[i])

# Print out the actual clusters
for label, cluster in clusters.items():
    print(f"Cluster {label}:")
    for statement in cluster:
        print(f"  {statement}")
    print()


5764
